[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/07_batchnorm.ipynb)

# 🟡 Medium: Implement BatchNorm

Implement **Batch Normalization** with both **training** and **inference** behavior.

In training mode, use **batch statistics** and update running estimates:

$$\text{BN}(x) = \gamma \cdot \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} + \beta$$

where $\mu_B$ and $\sigma_B^2$ are the mean and variance computed **across the batch** (dim=0).

In inference mode, use the provided **running mean/var** instead of current batch stats.

### Signature
```python
def my_batch_norm(
    x: torch.Tensor,
    gamma: torch.Tensor,
    beta: torch.Tensor,
    running_mean: torch.Tensor,
    running_var: torch.Tensor,
    eps: float = 1e-5,
    momentum: float = 0.1,
    training: bool = True,
) -> torch.Tensor:
    # x: (N, D) — normalize each feature across all samples in the batch
    # running_mean, running_var: updated in-place during training; used as-is during inference
```

### Rules
- Do **NOT** use `F.batch_norm`, `nn.BatchNorm1d`, etc.
- Compute batch mean and variance over `dim=0` with `unbiased=False`
- Update running stats like PyTorch: `running = (1 - momentum) * running + momentum * batch_stat`
- Use `running_mean` / `running_var` for inference when `training=False`
- Must support autograd w.r.t. `x`, `gamma`, `beta` (running statistics should be treated as buffers, not parameters requiring gradients)

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.2 MB/s eta 0:00:00


In [2]:
import torch

In [19]:
# ✏️ YOUR IMPLEMENTATION HERE

def my_batch_norm(
    x,
    gamma,
    beta,
    running_mean,
    running_var,
    eps=1e-5,
    momentum=0.1,
    training=True,
):
    # 统计量的更新不需要参与计算图的梯度反向传播
    with torch.no_grad():
        if training:
            # 1. 计算当前 Batch 的均值和方差
            m = x.mean(dim=0, keepdim=True)
            s = torch.mean((x - m) ** 2, dim=0, keepdim=True) # 去掉了结尾的 ** 2

            # 2. 原地更新 (In-place update) 全局 Buffer
            running_mean.copy_((1 - momentum) * running_mean + momentum * m.squeeze(0))
            running_var.copy_((1 - momentum) * running_var + momentum * s.squeeze(0))

            mean_to_use = m
            var_to_use = s
        else:
            # 推理时使用全局统计量，并增加一个维度以进行 Broadcasting 减法
            mean_to_use = running_mean.unsqueeze(0)
            var_to_use = running_var.unsqueeze(0)

    # 3. 归一化与仿射变换 (不论训练还是推理，公式统一，且都包含平方根)
    y = (x - mean_to_use) / torch.sqrt(var_to_use + eps)

    return gamma * y + beta

In [20]:
# 🧪 Debug
x = torch.randn(8, 4)
gamma = torch.ones(4)
beta = torch.zeros(4)

# Running stats typically live on the same device and shape as features
running_mean = torch.zeros(4)
running_var = torch.ones(4)

# Training mode: uses batch stats and updates running_mean / running_var
out_train = my_batch_norm(x, gamma, beta, running_mean, running_var, training=True)
print("[Train] Output shape:", out_train.shape)
print("[Train] Column means:", out_train.mean(dim=0))   # should be ~0
print("[Train] Column stds: ", out_train.std(dim=0))    # should be ~1
print("Updated running_mean:", running_mean)
print("Updated running_var:", running_var)

# Inference mode: uses running_mean / running_var only
out_eval = my_batch_norm(x, gamma, beta, running_mean, running_var, training=False)
print("[Eval] Output shape:", out_eval.shape)

[Train] Output shape: torch.Size([8, 4])
[Train] Column means: tensor([-8.9407e-08,  7.4506e-09, -5.5879e-09, -1.4901e-08])
[Train] Column stds:  tensor([1.0690, 1.0690, 1.0690, 1.0690])
Updated running_mean: tensor([ 0.0642,  0.0594,  0.0314, -0.0385])
Updated running_var: tensor([0.9980, 0.9470, 1.0242, 0.9841])
[Eval] Output shape: torch.Size([8, 4])


In [21]:
# ✅ SUBMIT
from torch_judge import check
check("batchnorm")


🧪 Testing: Implement BatchNorm (Medium)
──────────────────────────────────────────────────
  ✅ [1/4] Training mode — zero mean per feature (2.6ms)
  ✅ [2/4] Training mode — numerical correctness and running stats update (3.4ms)
  ✅ [3/4] Inference mode — uses running statistics (1.6ms)
  ✅ [4/4] Gradient flow w.r.t inputs and affine params (0.8ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (8.4ms total)
  Progress saved. Run status() to see your dashboard.

